# 🌿 Leafy Haven RAG Assistant
### A RAG-powered Q&A system extending the Leafy Haven rooftop greening application

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline that allows users to ask natural language questions about rooftop greening in Melbourne. Answers are grounded in a curated knowledge base of real guidelines, costs, permits, and maintenance advice.

**Tech Stack:** LangChain · ChromaDB · HuggingFace Embeddings · Llama 3.1 (Groq) · Gradio


## Step 1 — Install Dependencies
Install all required libraries for RAG pipeline, vector storage, embeddings, LLM access, and the chat interface.

In [ ]:
# Install all required packages
# langchain: RAG orchestration framework
# chromadb: local vector database for storing embeddings
# sentence-transformers: HuggingFace embedding model
# groq: free LLM API (Llama 3.1)
# gradio: chat interface
!pip install langchain==0.2.16 langchain-community==0.2.16 langchain-huggingface langchain-chroma langchain-groq langchain-text-splitters chromadb sentence-transformers groq gradio pypdf -q

## Step 2 — Restart Runtime
After installing new packages, the runtime must be restarted for Python to recognise them.

Continue from Step 3 after the restart.

In [ ]:
# Force restart the Colab runtime to load newly installed packages
import os
os.kill(os.getpid(), 9)

## Step 3 — Import Libraries

Start here after the runtime restart.

In [ ]:
import os

# RAG orchestration and retrieval chain
from langchain.chains import RetrievalQA

# HuggingFace embedding model — converts text to vectors
from langchain_huggingface import HuggingFaceEmbeddings

# ChromaDB — local vector database for storing and searching embeddings
from langchain_chroma import Chroma

# Text splitter — breaks documents into chunks for embedding
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Document loaders — reads PDF and text files into LangChain format
from langchain_community.document_loaders import PyPDFLoader, TextLoader

# Groq LLM — free API access to Llama 3.1
from langchain_groq import ChatGroq

# Gradio — builds the chat interface
import gradio as gr

print("All imports successful ✓")

## Step 4 — Connect to LLM
We use Llama 3.1 8B via Groq's free API for answer generation.
Get your free API key at [groq.com](https://groq.com).

In [ ]:
import os

# Set your Groq API key — get a free key at groq.com
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"

# Initialise the LLM
# model: Llama 3.1 8B — fast and capable, available on Groq free tier
# temperature: 0.2 — low temperature for factual, consistent answers
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)

# Test the connection
response = llm.invoke("Say hello in one sentence")
print(response.content)

## Step 5 — Build Knowledge Base
Create all 8 curated text documents that form the RAG knowledge base.
Each document covers a different aspect of rooftop greening that Leafy Haven users need to know.

| Document | Content |
|---|---|
| plant_selection.txt | Suitable plants for Melbourne rooftops |
| installation_costs.txt | Green roof types and costs per m² |
| permits_regulations.txt | Melbourne council permits and grants |
| green_score_explained.txt | How the Leafy Haven score is calculated |
| maintenance_benefits.txt | Seasonal maintenance and environmental benefits |
| visualisation_guide.txt | How to interpret the Stable Diffusion visualisation output |
| next_steps.txt | Recommended actions by score range with timelines |
| faq.txt | Common user questions about the Leafy Haven application |

In [ ]:
import os

# Create the documents folder if it doesn't exist
os.makedirs("documents", exist_ok=True)

# doc1: Plant selection guide
# Covers suitable plants for Melbourne rooftops including natives, succulents and herbs
doc1 = """
ROOFTOP GREENING GUIDE - PLANT SELECTION

Suitable Plants for Melbourne Rooftop Gardens:

SEDUMS AND SUCCULENTS:
- Sedum spurium: Drought-resistant, handles full sun, weight: very light, ideal for extensive green roofs
- Sedum acre: Low maintenance, spreads easily, excellent for flat rooftops
- Echeveria: Handles dry conditions, minimal soil depth required (5-10cm)

NATIVE AUSTRALIAN PLANTS:
- Lomandra longifolia: Hardy grass, drought tolerant, suits Melbourne climate
- Dianella revoluta: Low water needs, handles wind exposure well
- Scaevola aemula: Compact, handles heat and wind, good for exposed rooftops

HERBS AND EDIBLES:
- Thyme: Drought tolerant, aromatic, suits shallow soil (10-15cm)
- Rosemary: Handles full sun and wind, minimal maintenance
- Mint: Needs more water, suitable for irrigated rooftop beds

PLANTS TO AVOID:
- Large trees: Too heavy, root damage risk
- Plants needing deep soil (>30cm) for extensive roofs
- High water demand plants without irrigation system
"""

# doc2: Installation costs
# Covers extensive, intensive and semi-intensive green roof types with AUD costs
doc2 = """
ROOFTOP GREENING GUIDE - INSTALLATION AND COSTS

GREEN ROOF TYPES:

EXTENSIVE GREEN ROOF:
- Soil depth: 5-15cm
- Weight load: 60-150 kg/m²
- Cost: AUD $150-300 per m²
- Maintenance: Low (1-2 times per year)
- Best for: Sedums, grasses, herbs
- Suitable for most residential rooftops

INTENSIVE GREEN ROOF:
- Soil depth: 15-100cm
- Weight load: 180-500 kg/m²
- Cost: AUD $300-800 per m²
- Maintenance: High (regular watering, pruning)
- Best for: Shrubs, vegetables, small trees
- Requires structural engineer assessment

SEMI-INTENSIVE GREEN ROOF:
- Soil depth: 10-25cm
- Weight load: 120-200 kg/m²
- Cost: AUD $200-400 per m²
- Maintenance: Moderate
- Best for: Mix of sedums and small shrubs

INSTALLATION STEPS:
1. Structural assessment by engineer (AUD $500-1500)
2. Waterproofing membrane installation
3. Root barrier layer
4. Drainage layer
5. Filter fabric
6. Growing medium/substrate
7. Plant installation
"""

# doc3: Permits and regulations
# Covers City of Melbourne planning permits, building approvals and sustainability grants
doc3 = """
ROOFTOP GREENING GUIDE - PERMITS AND REGULATIONS MELBOURNE

CITY OF MELBOURNE REQUIREMENTS:

PLANNING PERMITS:
- Green roofs under 10m² generally do not require a planning permit
- Green roofs over 10m² may require a planning permit depending on zoning
- Contact Melbourne City Council on 03 9658 9658 for specific advice
- Apply through Victorian Planning Authority portal: planning.vic.gov.au

BUILDING PERMITS:
- Any structural modifications require a building permit
- Submit to your local council building department
- Require engineer certificate for loads over 150 kg/m²
- Typical processing time: 4-8 weeks

STRATA AND BODY CORPORATE:
- Apartment rooftops require body corporate approval
- Submit written proposal with structural engineer report
- Allow 4-12 weeks for approval process

HERITAGE OVERLAYS:
- Properties in heritage overlays require Heritage Victoria approval
- Additional assessments may be needed
- Contact Heritage Victoria: heritage.vic.gov.au

SUSTAINABILITY INCENTIVES:
- City of Melbourne Green Your Laneway grants: up to AUD $5000
- Victorian Government Greener Government Buildings program
- Some councils offer rate rebates for green infrastructure
"""

# doc4: Green score explained
# Covers how Leafy Haven calculates and interprets the 0-100 green score
doc4 = """
ROOFTOP GREENING GUIDE - GREEN SCORE EXPLAINED

LEAFY HAVEN GREEN SCORE SYSTEM:

The green score rates rooftop greening potential from 0-100.

SCORE COMPONENTS:
1. Existing Vegetation Coverage (25 points)
   - 0-10% coverage: 0-8 points
   - 10-30% coverage: 8-16 points
   - 30%+ coverage: 16-25 points

2. Rooftop Area (25 points)
   - Under 20m²: 0-8 points
   - 20-50m²: 8-16 points
   - Over 50m²: 16-25 points

3. Sun Exposure (25 points)
   - North-facing (best in Melbourne): 20-25 points
   - East/West-facing: 12-18 points
   - South-facing (least sun): 0-10 points

4. Structural Suitability (25 points)
   - Flat roof: 20-25 points
   - Slight slope (<10 degrees): 12-18 points
   - Steep slope (>10 degrees): 0-10 points

SCORE INTERPRETATION:
- 0-25: Low potential — structural or sun exposure challenges
- 26-50: Moderate potential — extensive green roof recommended
- 51-75: Good potential — semi-intensive or extensive roof suitable
- 76-100: Excellent potential — full green roof highly recommended

IMPROVING YOUR SCORE:
- Add irrigation system to support more plant variety
- Install lightweight substrate to reduce structural load
- Consider vertical gardens if rooftop area is limited
"""

# doc5: Maintenance and environmental benefits
# Covers seasonal maintenance schedule and quantified environmental benefits
doc5 = """
ROOFTOP GREENING GUIDE - MAINTENANCE AND ENVIRONMENTAL BENEFITS

MAINTENANCE SCHEDULE:

SPRING (September-November):
- Inspect waterproofing membrane
- Fertilise with slow-release organic fertiliser
- Plant new seedlings
- Check and clear drainage points

SUMMER (December-February):
- Water 2-3 times per week without irrigation system
- Monitor for pest activity
- Trim overgrown plants
- Check for heat stress in plants

AUTUMN (March-May):
- Reduce watering frequency
- Remove dead plant material
- Inspect root barriers
- Prepare for winter with mulching

WINTER (June-August):
- Minimal maintenance needed
- Check drainage after heavy rain
- Melbourne average winter rainfall: 150mm — usually sufficient

ENVIRONMENTAL BENEFITS:
- Temperature reduction: Green roofs reduce rooftop temperature by 20-40°C
- Stormwater management: Absorbs 50-90% of rainfall
- Urban heat island: Can reduce surrounding air temperature by 1-3°C
- Biodiversity: Provides habitat for birds and insects
- Air quality: Filters particulate matter and CO2
- Building energy: Reduces cooling costs by 15-25%
- Noise reduction: 8-15 decibel reduction in building noise

WATER USAGE:
- Extensive roof: 2-4 litres/m²/week in summer
- Intensive roof: 8-15 litres/m²/week in summer
- Rainwater harvesting recommended to offset usage
"""

# doc6: Visualisation guide
# Explains how to interpret the Stable Diffusion green overlay output
doc6 = """
LEAFY HAVEN - UNDERSTANDING YOUR VISUALISATION RESULTS

WHAT THE STABLE DIFFUSION VISUALISATION SHOWS:
The green visualisation image shows a realistic preview of what your rooftop could look like after greening.
This is an AI-generated image using Stable Diffusion inpainting technology.

HOW TO INTERPRET YOUR VISUALISATION:
- Green overlay areas: Zones identified as suitable for plant installation
- Darker green areas: Higher density planting recommended
- Lighter green areas: Low maintenance ground cover recommended
- No overlay areas: Structurally unsuitable or already vegetated zones

IMPORTANT NOTES ABOUT THE VISUALISATION:
- The image is an AI preview only, not an architectural plan
- Actual plant species and layout should be confirmed with a landscape architect
- The visualisation assumes standard soil depth of 10cm
- Colours in the visualisation represent vegetation density, not specific plant species

WHAT TO DO AFTER SEEING YOUR VISUALISATION:
1. Note your green score and save your visualisation image
2. Share with a landscape architect or green roof installer for professional advice
3. Contact your local council to check permit requirements
4. Get 2-3 quotes from certified green roof installers
5. Consider joining the City of Melbourne Urban Greening program
"""

# doc7: Next steps by score range
# Covers recommended actions for each score band with timelines and contacts
doc7 = """
LEAFY HAVEN - NEXT STEPS AFTER YOUR ROOFTOP ANALYSIS

SCORE 0-25 (LOW POTENTIAL):
- Your rooftop may have structural or sun exposure challenges
- Consider vertical gardens on walls or balconies instead
- Explore indoor plant walls as an alternative
- Contact a structural engineer to assess improvement options
- Estimated cost to improve suitability: AUD $2000-5000

SCORE 26-50 (MODERATE POTENTIAL):
- Extensive green roof is your best option
- Start small with a test area of 5-10m²
- Recommended first plants: Sedums and native grasses
- Get a structural assessment before proceeding
- Expected installation timeline: 4-8 weeks after permits
- Connect with: City of Melbourne Green Roof Network

SCORE 51-75 (GOOD POTENTIAL):
- Semi-intensive or extensive roof both viable
- Consider a mix of ornamental and edible plants
- Irrigation system recommended for best results
- Apply for City of Melbourne greening grants
- Engage a landscape architect for design
- Expected installation timeline: 6-10 weeks after permits

SCORE 76-100 (EXCELLENT POTENTIAL):
- Full green roof highly recommended
- Both intensive and extensive options available
- Opportunity to create a rooftop garden or urban farm
- High value addition to property
- Eligible for maximum government incentives
- Consider rooftop beekeeping or composting additions
- Expected installation timeline: 8-16 weeks after permits

FINDING PROFESSIONAL HELP IN MELBOURNE:
- Green Roofs Australasia: greenroofs.org.au
- Landscape Architecture Australia: landscapeaustralia.com.au
- City of Melbourne Urban Forest: melbourne.vic.gov.au/urbanforest
"""

# doc8: FAQ
# Covers common user questions about the Leafy Haven application
doc8 = """
LEAFY HAVEN - FREQUENTLY ASKED QUESTIONS

Q: How accurate is the green score?
A: The green score is based on computer vision analysis of your uploaded image. It analyses vegetation coverage, rooftop area, sun exposure and structural suitability. Accuracy is highest with clear, high resolution aerial or rooftop images taken in daylight. The score is a guide only and should be validated by a professional assessor.

Q: What type of image should I upload to Leafy Haven?
A: For best results upload a clear aerial or top-down photograph of your rooftop taken in daylight. Avoid images with heavy shadows, obstructions or low resolution. Google Maps satellite view screenshots also work well.

Q: Can I use Leafy Haven for any rooftop in Australia?
A: Leafy Haven is optimised for Melbourne rooftops and uses Melbourne-specific climate data, council regulations and plant databases. Results for rooftops outside Melbourne may be less accurate for permit and plant recommendations.

Q: Is my rooftop image stored or shared?
A: Images uploaded to Leafy Haven are processed in real time and are not stored permanently or shared with third parties.

Q: Can Leafy Haven analyse multiple rooftops at once?
A: Currently Leafy Haven analyses one rooftop per submission. For multiple rooftops submit each image separately.

Q: What if my rooftop has existing vegetation?
A: Leafy Haven detects existing vegetation using computer vision and factors this into your green score. Existing vegetation positively contributes to your score.

Q: How often should I re-analyse my rooftop?
A: Re-analyse after any major changes to the rooftop structure, after installing new vegetation, or annually to track your greening progress over time.

Q: What is the difference between Leafy Haven and a landscape architect?
A: Leafy Haven provides an AI-powered preliminary assessment to help you understand your rooftop's potential quickly and for free. A landscape architect provides detailed professional design, engineering sign-off and project management for actual installation.
"""

# Map all 8 filenames to their document content
docs = {
    "plant_selection.txt": doc1,
    "installation_costs.txt": doc2,
    "permits_regulations.txt": doc3,
    "green_score_explained.txt": doc4,
    "maintenance_benefits.txt": doc5,
    "visualisation_guide.txt": doc6,
    "next_steps.txt": doc7,
    "faq.txt": doc8
}

# Write all 8 documents to the documents/ folder
for filename, content in docs.items():
    with open(f"documents/{filename}", "w") as f:
        f.write(content)

print(f"✓ Created {len(docs)} knowledge base documents")
for filename in docs.keys():
    print(f"  - {filename}")

## Step 6 — Chunk, Embed and Store in ChromaDB
This is the core of the RAG pipeline. Three things happen here:

1. **Load** — all 8 documents are read from the `documents/` folder
2. **Chunk** — each document is split into 500-character overlapping chunks so the retriever can find precise answers
3. **Embed and Store** — each chunk is converted to a vector using HuggingFace sentence-transformers and stored in ChromaDB for semantic search

> Why chunk? LLMs have context limits. Chunking lets us retrieve only the most relevant pieces rather than sending entire documents to the LLM every time.

In [ ]:
# Import document loaders, text splitter, embeddings and vector store
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# --- STEP 1: LOAD ---
# Read all .txt files from the documents/ folder into LangChain Document objects
print("Loading documents...")
documents = []
for filename in os.listdir("documents"):
    if filename.endswith(".txt"):
        loader = TextLoader(f"documents/{filename}")
        documents.extend(loader.load())
        print(f"  ✓ Loaded {filename}")

# --- STEP 2: CHUNK ---
# Split documents into overlapping chunks of 500 characters
# chunk_overlap=50 ensures context is not lost at chunk boundaries
print("\nChunking documents...")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,    # maximum characters per chunk
    chunk_overlap=50   # overlap between consecutive chunks to preserve context
)
chunks = splitter.split_documents(documents)
print(f"  ✓ Created {len(chunks)} chunks from {len(documents)} documents")

# --- STEP 3: EMBED AND STORE ---
# Load the HuggingFace embedding model
# all-MiniLM-L6-v2: lightweight, fast, and effective for semantic similarity
print("\nLoading embedding model (first time may take 1-2 mins)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("  ✓ Embedding model loaded")

# Convert all chunks to vectors and store in ChromaDB
# ChromaDB persists the vector store locally so it can be reloaded later
print("\nStoring in ChromaDB...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"  # local folder for vector storage
)
print(f"  ✓ Stored {len(chunks)} chunks in ChromaDB")
print("\n✓ Knowledge base ready!")

## Step 7 — Build the RAG Chain
This step connects all the components into a single retrieval and generation pipeline.

**How it works:**
1. User submits a question
2. Question is converted to a vector using the same embedding model
3. ChromaDB finds the 3 most semantically similar chunks (`k=3`)
4. Those chunks are injected into the prompt as context
5. Llama 3.1 reads the context and generates a grounded answer

> The `stuff` chain type means all retrieved chunks are stuffed directly into the prompt, simple and effective for small knowledge bases like ours.

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Reconnect to the existing ChromaDB vector store
# Uses the same embedding model to ensure vectors are compatible
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

# Create the retriever
# search_type="similarity": uses cosine similarity to find relevant chunks
# k=3: retrieve the 3 most relevant chunks for each question
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Custom prompt template
# {context}: retrieved chunks are injected here
# {question}: the user's question is injected here
# The fallback instruction prevents hallucination on out-of-scope questions
prompt_template = """You are a helpful assistant for Leafy Haven, an AI rooftop greening application for Melbourne, Australia.
Use the following information to answer the user's question accurately and helpfully.
If the answer is not in the provided information, say "I don't have specific information about that, please contact the City of Melbourne council for guidance."

Context:
{context}

Question: {question}

Helpful Answer:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Build the RAG chain
# llm: Llama 3.1 via Groq for answer generation
# chain_type="stuff": injects all retrieved chunks directly into the prompt
# return_source_documents=True: returns which documents were used for each answer
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)

print("✓ RAG chain ready!")

# Quick test to verify the full pipeline is working end-to-end
print("\nTesting RAG chain...")
result = rag_chain.invoke({"query": "What does a green score of 65 mean for my rooftop?"})
print("\nAnswer:", result["result"])

## Step 8 — Launch the Chat Interface
Builds and launches the Gradio chat application with:
- A chat window that maintains conversation history
- Source citation on every answer showing which documents were used
- 10 example questions covering all knowledge base topics
- A public shareable link valid for 72 hours

> The `share=True` parameter generates a public gradio.live URL.

In [ ]:
import gradio as gr

def ask_leafy_haven(question, history):
    if not question.strip():
        return "Please ask a question about rooftop greening!"

    result = rag_chain.invoke({"query": question})
    answer = result["result"]

    # Show which documents were used
    sources = set()
    for doc in result["source_documents"]:
        source = doc.metadata.get("source", "").replace("documents/", "").replace(".txt", "").replace("_", " ").title()
        sources.add(source)

    if sources:
        answer += f"\n\n📚 *Sources: {', '.join(sources)}*"

    return answer

# Build the interface
with gr.Blocks(theme=gr.themes.Soft(), title="Leafy Haven Assistant") as app:
    gr.Markdown("""
    # 🌿 Leafy Haven AI Assistant
    ### Your rooftop greening guide for Melbourne
    Ask me anything about green scores, plant selection, installation costs, permits, maintenance, visualisation results, or next steps after your analysis!
    """)

    chatbot = gr.Chatbot(height=400, placeholder="Ask me about your rooftop greening project...")

    with gr.Row():
        msg = gr.Textbox(
            placeholder="e.g. What plants suit a north-facing rooftop?",
            scale=4,
            show_label=False
        )
        submit = gr.Button("Ask 🌱", scale=1, variant="primary")

    gr.Examples(
        examples=[
            "What does a green score of 45 mean?",
            "What plants work best for Melbourne rooftops?",
            "How much does a green roof cost per square metre?",
            "Do I need a permit for rooftop greening in Melbourne?",
            "How do I maintain a green roof in summer?",
            "What are the environmental benefits of rooftop greening?",
            "What does the green overlay in my visualisation mean?",
            "I got a score of 80, what should I do next?",
            "Is my uploaded image stored by Leafy Haven?",
            "How is Leafy Haven different from a landscape architect?"
        ],
        inputs=msg
    )

    def respond(message, chat_history):
        answer = ask_leafy_haven(message, chat_history)
        chat_history.append((message, answer))
        return "", chat_history

    submit.click(respond, [msg, chatbot], [msg, chatbot])
    msg.submit(respond, [msg, chatbot], [msg, chatbot])

app.launch(share=True)